In [1]:

!pip install chromadb sentence-transformers groq -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curre

In [2]:
# Mount Google Drive and load Groq API key
from google.colab import drive, userdata

# Mount GDrive
drive.mount('/content/drive')

# Load Groq API key from Colab secrets
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Drive mounted.")
print("Groq API key loaded:", os.environ["GROQ_API_KEY"][:8], "...")

Mounted at /content/drive
Drive mounted.
Groq API key loaded: gsk_vGTj ...


In [3]:
# Load existing ChromaDB and embedding model
import chromadb
from sentence_transformers import SentenceTransformer

# Load the same embedding model used during storage
# MUST be identical to what you used in Day 3 — any difference breaks retrieval
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.")

# Connect to your persistent ChromaDB from GDrive
chroma_client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/rag_day3/chroma_store"
)

# List available collections — so we know the exact collection name
collections = chroma_client.list_collections()
print("Collections found:")
for col in collections:
    print(" →", col.name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.
Collections found:
 → wiki_docs


In [4]:
# Load the wiki_docs collection and inspect what's inside
collection = chroma_client.get_collection(name="wiki_docs")

# How many chunks are stored?
print("Total chunks stored:", collection.count())

# Peek at the first 3 chunks to confirm content looks right
sample = collection.peek(limit=3)

print("\nSample chunks:")
for i, doc in enumerate(sample["documents"]):
    print(f"\n--- Chunk {i+1} ---")
    print(doc[:300])  # First 300 chars only
    print("Metadata:", sample["metadatas"][i])

Total chunks stored: 252

Sample chunks:

--- Chunk 1 ---
In deep learning, the transformer is a family of artificial neural network architectures based on the multi-head attention mechanism, in which text is converted to numerical representations called tokens, and each token is converted into a vector via lookup from a word embedding table. At each layer
Metadata: None

--- Chunk 2 ---
 allowing the signal for key tokens to be amplified and less important tokens to be diminished. 
Transformers have the advantage of having no recurrent units, therefore requiring less training time than earlier recurrent neural architectures (RNNs) such as long short-term memory (LSTM). Later variat
Metadata: None

--- Chunk 3 ---
cture was proposed in the 2017 paper "Attention Is All You Need" by researchers at Google. The predecessors of transformers were developed as an improvement over previous architectures for machine translation, but have found many applications since. They are used in large-sca

In [6]:
# Cell 5 (Fixed): Handle None metadata gracefully

def retrieve_chunks(query: str, top_k: int = 3) -> list[dict]:
    """
    Embeds the query and fetches top_k chunks from ChromaDB.
    Handles missing/None metadata safely.
    """

    # Step 1: Embed the query
    query_vector = embedding_model.encode(query).tolist()

    # Step 2: Query ChromaDB
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    # Step 3: Package results — safely handle None metadata
    chunks = []
    for i in range(len(results["documents"][0])):
        metadata = results["metadatas"][0][i]  # could be None

        chunks.append({
            "chunk_id": i,
            "source": metadata.get("source", "unknown") if metadata else "unknown",
            "text": results["documents"][0][i],
            "distance": round(results["distances"][0][i], 4)
        })

    return chunks


# Test
query = "What is the attention mechanism in a Transformer?"
retrieved = retrieve_chunks(query, top_k=3)

print(f"Query: {query}\n")
for chunk in retrieved:
    print(f"[Chunk {chunk['chunk_id']+1}] Source: {chunk['source']} | Distance: {chunk['distance']}")
    print(chunk['text'][:300])
    print()

Query: What is the attention mechanism in a Transformer?

[Chunk 1] Source: unknown | Distance: 0.3305
   
        
      
    
    {\displaystyle c_{j}}
  
. This allows the transformer to take any encoded position and find a linear sum of the encoded locations of its neighbors. This sum of encoded positions, when fed into the attention mechanism, would create attention weights on its neighbors, muc

[Chunk 2] Source: unknown | Distance: 0.3313
mparing the behavior of transformer architectures over long inputs.

Alternative attention graphs
The standard attention graph is either all-to-all or causal, both of which scales as 
  
    
      
        O
        (
        
          N
          
            2
          
        
        )
     

[Chunk 3] Source: unknown | Distance: 0.3803
on, the scope of attention, or the range of token relationships captured by each attention head, can expand as tokens pass through successive layers. This allows the model to capture more complex and lon

In [7]:
# Build a grounded prompt from retrieved chunks

def build_prompt(query: str, retrieved_chunks: list[dict]) -> str:
    """
    Constructs a grounded prompt.
    Forces the model to answer ONLY from retrieved context.
    Requires citations in the answer.
    """

    # Format each chunk with a numbered reference label
    context_block = ""
    for i, chunk in enumerate(retrieved_chunks):
        context_block += f"[{i+1}] Source: {chunk['source']}\n"
        context_block += f"{chunk['text'].strip()}\n\n"

    prompt = f"""You are a factual assistant. Answer the question using ONLY the context provided below.
If the answer is not present in the context, respond with: "I don't have enough information to answer this."
Do NOT use your own knowledge. Cite the source numbers (e.g. [1], [2]) for every claim you make.

CONTEXT:
{context_block}
QUESTION: {query}

ANSWER (with citations):"""

    return prompt


# Test — print the full prompt so you can see exactly what goes to the model
prompt = build_prompt(query, retrieved)
print(prompt)

You are a factual assistant. Answer the question using ONLY the context provided below.
If the answer is not present in the context, respond with: "I don't have enough information to answer this."
Do NOT use your own knowledge. Cite the source numbers (e.g. [1], [2]) for every claim you make.

CONTEXT:
[1] Source: unknown
{\displaystyle c_{j}}
  
. This allows the transformer to take any encoded position and find a linear sum of the encoded locations of its neighbors. This sum of encoded positions, when fed into the attention mechanism, would create attention weights on its neighbors, much like what happens in a convolutional neural network language model. In the author's words, "we hypothesized it would allow the model to easily learn to attend by relative position."
In typical imple

[2] Source: unknown
mparing the behavior of transformer architectures over long inputs.

Alternative attention graphs
The standard attention graph is either all-to-all or causal, both of which scales as 

In [8]:
# Generate answer using Groq API

from groq import Groq

# Initialize Groq client — picks up key from os.environ set in Cell 2
groq_client = Groq()

def generate_answer(prompt: str, temperature: float = 0.1) -> str:
    """
    Sends the grounded prompt to Groq's LLaMA model.
    Low temperature = less creative = less hallucination.
    """

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",   # Best available on Groq free tier
        messages=[
            {
                "role": "system",
                "content": "You are a factual assistant that only answers from provided context."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,      # Low = stick to context, don't improvise
        max_tokens=512,
    )

    return response.choices[0].message.content


# Test generation
answer = generate_answer(prompt)
print("ANSWER:\n")
print(answer)

ANSWER:

The attention mechanism in a Transformer allows the model to take any encoded position and find a linear sum of the encoded locations of its neighbors, creating attention weights on its neighbors [1]. This is similar to what happens in a convolutional neural network language model. The standard attention graph is either all-to-all or causal, scaling as O(N^2) where N is the number of tokens in a sequence [2]. The attention mechanism can capture complex and long-range dependencies in deeper layers, with some attention heads encoding relevance relations that are meaningful to humans, such as attending to the next word or from verbs to their direct objects [3].


In [9]:
# Extract and display citations from the generated answer

import re

def extract_citations(answer: str, retrieved_chunks: list[dict]) -> list[dict]:
    """
    Parses [1], [2], [3] style citations from the answer.
    Maps them back to the actual chunk text.
    """

    # Find all citation numbers used in the answer
    raw = re.findall(r'\[(\d+)\]', answer)
    cited_indices = sorted(set(int(n) for n in raw))

    citations = []
    for idx in cited_indices:
        chunk_index = idx - 1  # Convert 1-based to 0-based
        if 0 <= chunk_index < len(retrieved_chunks):
            citations.append({
                "reference": f"[{idx}]",
                "source": retrieved_chunks[chunk_index]["source"],
                "distance": retrieved_chunks[chunk_index]["distance"],
                "excerpt": retrieved_chunks[chunk_index]["text"][:200].strip() + "..."
            })

    return citations


# Test
citations = extract_citations(answer, retrieved)

print("CITED SOURCES:\n")
for c in citations:
    print(f"{c['reference']} → Source: {c['source']} | Retrieval Distance: {c['distance']}")
    print(f"Excerpt: {c['excerpt']}")
    print()

CITED SOURCES:

[1] → Source: unknown | Retrieval Distance: 0.3305
Excerpt: {\displaystyle c_{j}}
  
. This allows the transformer to take any encoded position and find a linear sum of the encoded locations of its neighbors. This sum of encoded po...

[2] → Source: unknown | Retrieval Distance: 0.3313
Excerpt: mparing the behavior of transformer architectures over long inputs.

Alternative attention graphs
The standard attention graph is either all-to-all or causal, both of which scales as...

[3] → Source: unknown | Retrieval Distance: 0.3803
Excerpt: on, the scope of attention, or the range of token relationships captured by each attention head, can expand as tokens pass through successive layers. This allows the model to capture more complex and...



In [10]:
# Complete RAG generation pipeline — single entry point

def rag_generate(query: str, top_k: int = 3) -> dict:
    """
    Full pipeline:
    1. Retrieve relevant chunks from ChromaDB
    2. Build grounded prompt
    3. Generate answer via Groq
    4. Extract citations
    Returns everything as a structured dict.
    """

    # Step 1: Retrieve
    retrieved_chunks = retrieve_chunks(query, top_k=top_k)

    # Step 2: Build prompt
    prompt = build_prompt(query, retrieved_chunks)

    # Step 3: Generate
    answer = generate_answer(prompt)

    # Step 4: Extract citations
    citations = extract_citations(answer, retrieved_chunks)

    return {
        "query": query,
        "answer": answer,
        "citations": citations,
        "chunks_retrieved": len(retrieved_chunks)
    }


# ── Test with 3 different queries ──────────────────────────────────────────────

queries = [
    "How does multi-head attention work?",
    "What is positional encoding in Transformers?",
    "What are the limitations of the standard attention mechanism?"
]

for q in queries:
    print("=" * 60)
    result = rag_generate(q)
    print(f"QUERY: {result['query']}")
    print(f"\nANSWER:\n{result['answer']}")
    print(f"\nCITED SOURCES:")
    for c in result['citations']:
        print(f"  {c['reference']} → {c['source']} (distance: {c['distance']})")
    print()

QUERY: How does multi-head attention work?

ANSWER:
Multi-head attention allows for fast processing as the computations for each attention head can be performed in parallel [1]. The outputs for the attention layer are concatenated to pass into the feedforward neural network layers [1]. Additionally, each attention head can have a different head dimension, but this is rarely the case in practice [2]. The scope of attention can expand as tokens pass through successive layers, allowing the model to capture more complex and long-range dependencies [3]. 

Note: The exact mathematical formulation of multi-head attention is partially provided as MultiheadAttention(Q, K, V) [1], but the details of the formulation are not fully specified in the provided context.

CITED SOURCES:
  [1] → unknown (distance: 0.2718)
  [2] → unknown (distance: 0.3153)
  [3] → unknown (distance: 0.3821)

QUERY: What is positional encoding in Transformers?

ANSWER:
A positional encoding is a fixed-size vector represen